# RAG Sprint 1 – מנוע חיפוש סמנטי על "מדריך לרוכש דירה"

נוטבוק זה בונה את שכבת ה־**Retrieval** של מערכת RAG, שלב אחר שלב:

1. **טעינת המסמך** – קריאת ה־PDF והמרתו לטקסט.
2. **חלוקה ל־Chunks** – פירוק הטקסט לקטעים קטנים וחופפים.
3. **Embeddings** – המרת כל chunk לוקטור מספרים באמצעות `gemini-embedding-001`.
4. **Pinecone** – שמירת הוקטורים במסד נתונים וקטורי.
5. **Semantic Search** – חיפוש לפי משמעות + הצגת התוצאות הרלוונטיות.

> כל שלב מתועד בתא Markdown שמסביר מה קורה ולמה.

## שלב 2 – טעינת המסמך וחלוקה ל־Chunks

**למה מחלקים ל־Chunks?**
מודל ה־Embeddings וה־Retrieval עובדים טוב יותר על קטעים קצרים וממוקדים מאשר על מסמך שלם.
בנוסף, יש הגבלת אורך לקלט. לכן מפרקים את הטקסט לקטעים ("chunks").

**מה זה `chunk_overlap`?**
כדי לא "לחתוך" משפט או רעיון באמצע בין שני chunks, אנחנו משאירים חפיפה של כמה תווים
בין chunk לחבירו. כך מידע שנמצא על הגבול עדיין מופיע בשלמותו באחד הקטעים.

In [1]:
# === תא הגדרות – הריצי את זה ראשון ===
import sys
import subprocess

try:
    import pip_system_certs.bootstrap
except ImportError:
    print("מתקין pip-system-certs...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pip-system-certs", "-q"])
    import pip_system_certs.bootstrap

import os
from pathlib import Path

from dotenv import load_dotenv
from pypdf import PdfReader

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

print("Python:", sys.executable)
print("Project root:", PROJECT_ROOT)
print("GEMINI_API_KEY:", "כן ✓" if GEMINI_API_KEY else "לא ✗ – צרי קובץ .env")
print("PINECONE_API_KEY:", "כן ✓" if PINECONE_API_KEY else "לא ✗ – צרי קובץ .env")

PDF_PATH = PROJECT_ROOT / "data" / "apartment_buyer_guide.pdf"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
EMBED_MODEL = "gemini-embedding-001"
EMBED_DIM = 768
INDEX_NAME = "rag-apartment-guide"

print("\nPDF:", PDF_PATH)
print("PDF קיים:", "כן ✓" if PDF_PATH.exists() else "לא ✗")

Python: C:\Users\WIN 11\Desktop\תהילה\יד\AI\RAG-system\.venv\Scripts\python.exe
Project root: C:\Users\WIN 11\Desktop\תהילה\יד\AI\RAG-system
GEMINI_API_KEY: כן ✓
PINECONE_API_KEY: כן ✓

PDF: C:\Users\WIN 11\Desktop\תהילה\יד\AI\RAG-system\data\apartment_buyer_guide.pdf
PDF קיים: כן ✓


In [2]:
import re


def clean_text(text):
    """ניקוי טקסט שחולץ מ-PDF: מכווץ רצפים ארוכים של רווחים ושורות ריקות
    (ה-PDF מכיל הרבה שורות ריקות שמוסיפות רעש ל-Embeddings)."""
    text = re.sub(r"[ \t]+", " ", text)      # רצף רווחים -> רווח בודד
    text = re.sub(r"\n\s*\n+", "\n", text)   # שורות ריקות מרובות -> שורה אחת
    return text.strip()


def load_pdf(pdf_path):
    """קורא PDF ומחזיר:
    - full_text: כל הטקסט של המסמך כמחרוזת אחת (מנוקה)
    - char_page: רשימה שממפה כל תו במיקום i למספר העמוד שממנו הגיע (לצורך metadata)
    """
    reader = PdfReader(str(pdf_path))
    full_text = ""
    char_page = []
    for page_num, page in enumerate(reader.pages, start=1):
        page_text = clean_text(page.extract_text() or "") + "\n"
        full_text += page_text
        char_page.extend([page_num] * len(page_text))
    return full_text, char_page


full_text, char_page = load_pdf(PDF_PATH)

print(f"מספר עמודים: {char_page[-1] if char_page else 0}")
print(f"סך תווים בטקסט: {len(full_text):,}")
print("\n--- תצוגה מקדימה (300 תווים ראשונים) ---")
print(full_text[:300])

מספר עמודים: 18
סך תווים בטקסט: 19,998

--- תצוגה מקדימה (300 תווים ראשונים) ---
1 
מדריך לרוכש דירה
2 
תוכן העניינים 
מבוא : מה צריך לבדוק לפני שרוכשים דירה?1 
פרק1 : הדירה ואזור המגוריםבחירת4 
פרק2 : בודקיםמה לפני הרכישה6 
פרק3 : יכרון דברים וחוזה המכרז8 
פרק4 : מהלך הבנייה11 
פרק5 : הבטחת הכספים12 
פרק6 : קבלת הדירה ואחריות מוכר הדירה15 
פרק7 : הגשת תלונות למשרד הבינוי והשיכו


In [4]:
def chunk_text(text, char_page, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
    """מחלק טקסט ארוך לקטעים בגודל chunk_size תווים, עם חפיפה של chunk_overlap.
    כל chunk מקבל metadata עם מספר העמוד שבו הוא מתחיל.
    מחזיר רשימת dict: {id, text, page}.
    """
    if chunk_overlap >= chunk_size:
        raise ValueError("chunk_overlap חייב להיות קטן מ-chunk_size")

    chunks = []
    step = chunk_size - chunk_overlap  # כמה מתקדמים כל פעם
    idx = 0
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk_str = text[start:end].strip()
        if chunk_str:  # מדלגים על קטעים ריקים
            page = char_page[start] if start < len(char_page) else char_page[-1]
            chunks.append({
                "id": f"chunk-{idx}",
                "text": chunk_str,
                "page": page,
            })
            idx += 1
        start += step
    return chunks

In [5]:
chunks = chunk_text(full_text, char_page, CHUNK_SIZE, CHUNK_OVERLAP)

print(f"נוצרו {len(chunks)} chunks (chunk_size={CHUNK_SIZE}, chunk_overlap={CHUNK_OVERLAP})")
print("\n--- דוגמה: ה-chunk הראשון ---")
print("id:", chunks[0]["id"], "| עמוד:", chunks[0]["page"])
print(chunks[0]["text"][:400], "...")

print("\n--- דוגמה: chunk מאמצע המסמך ---")
mid = len(chunks) // 2
print("id:", chunks[mid]["id"], "| עמוד:", chunks[mid]["page"])
print(chunks[mid]["text"][:400], "...")

נוצרו 24 chunks (chunk_size=1000, chunk_overlap=150)

--- דוגמה: ה-chunk הראשון ---
id: chunk-0 | עמוד: 1
1 
מדריך לרוכש דירה
2 
תוכן העניינים 
מבוא : מה צריך לבדוק לפני שרוכשים דירה?1 
פרק1 : הדירה ואזור המגוריםבחירת4 
פרק2 : בודקיםמה לפני הרכישה6 
פרק3 : יכרון דברים וחוזה המכרז8 
פרק4 : מהלך הבנייה11 
פרק5 : הבטחת הכספים12 
פרק6 : קבלת הדירה ואחריות מוכר הדירה15 
פרק7 : הגשת תלונות למשרד הבינוי והשיכון17 
פרק8 : הבהרות כלליות שימוש במדריך18
3 
צעדיו הראשונים לקראת רכישה של דירהדריך זה נכתב במיוחד לר ...

--- דוגמה: chunk מאמצע המסמך ---
id: chunk-12 | עמוד: 10
רישום של רשות מקרקעי בחוזה. כמו כן, מומלץ לבדוק אם החברה המוכרת מצויה ב
. ישראל
ליווי משפטי בעת תהליך רכישת הדירה 
רצוי לשכור את שירותיו של עורך דין מתחום הנדל"ן לצורך בדיקת כל פרטי הדירה וניהול המשא ומתן 
כמו כן, רצוי לחתום על חוזה המכר בנוכחות עורך הדין שנשכר מטעמכם. זכרו להחתים את . מול המוכר
 .המוכר גם על העתק החוזה שנשאר ברשותכם
בכל הנוגע לרישום הדירה שימו לב: המוכר שוכר עורך דין מטעמו אשר נו ...


## שלב 3 – יצירת Embeddings עם Gemini

**מה זה Embedding?**
Embedding הוא ייצוג מספרי (וקטור) של טקסט. טקסטים עם משמעות דומה מקבלים וקטורים "קרובים" במרחב.

**למה `task_type`?**
למודל `gemini-embedding-001` מומלץ להפריד:
- `RETRIEVAL_DOCUMENT` – לטקסטים שנשמרים בבסיס הידע (chunks)
- `RETRIEVAL_QUERY` – לשאלות של המשתמש (נשתמש בזה בשלב החיפוש)

**אם מקבלים שגיאת SSL (נפוץ ב-NetFree):**
1. ודאי שקיים קובץ `.env` (לא `.env.example`) עם המפתחות האמיתיים.
2. התקיני את תעודת השורש של NetFree במחשב.
3. אם עדיין נכשל – הריצי בטרמינל: `pip install pip-system-certs` ואז **Restart Kernel** בנוטבוק.

In [6]:
from google import genai
from google.genai import types

# יצירת לקוח Gemini עם המפתח מ-.env
# (המפתח נטען בתא ההגדרות למעלה)
if not GEMINI_API_KEY:
    raise ValueError(
        "חסר GEMINI_API_KEY. צרי קובץ .env בשורש הפרויקט:\n"
        "Copy-Item .env.example .env\n"
        "ואז מלאי את המפתח האמיתי ב-.env (לא ב-.env.example!)"
    )

client = genai.Client(api_key=GEMINI_API_KEY)
print("Gemini client נוצר בהצלחה")

Gemini client נוצר בהצלחה


In [7]:
import time

def embed_texts(texts, task_type="RETRIEVAL_DOCUMENT"):
    """ממיר רשימת טקסטים לוקטורים. כולל retry אוטומטי אם יש rate limit."""
    for attempt in range(3):
        try:
            result = client.models.embed_content(
                model=EMBED_MODEL,
                contents=texts,
                config=types.EmbedContentConfig(
                    task_type=task_type,
                    output_dimensionality=EMBED_DIM,
                ),
            )
            return [emb.values for emb in result.embeddings]
        except Exception as exc:
            if "429" in str(exc) and attempt < 2:
                print("Rate limit – ממתין 35 שניות...")
                time.sleep(35)
                continue
            raise

sample_vector = embed_texts([chunks[0]["text"]], task_type="RETRIEVAL_DOCUMENT")[0]
print(f"מימד הוקטור: {len(sample_vector)}")
print(f"3 ערכים ראשונים: {sample_vector[:3]}")

מימד הוקטור: 768
3 ערכים ראשונים: [0.00069829734, -0.003149922, 0.007863436]


In [8]:
# יצירת embeddings לכל ה-chunks (זה לוקח כמה שניות)
all_texts = [c["text"] for c in chunks]
all_vectors = embed_texts(all_texts, task_type="RETRIEVAL_DOCUMENT")

# שיוך הוקטור לכל chunk
for chunk, vector in zip(chunks, all_vectors):
    chunk["vector"] = vector

print(f"נוצרו embeddings ל-{len(all_vectors)} chunks")
print(f"דוגמה: {chunks[0]['id']} -> וקטור באורך {len(chunks[0]['vector'])}")

נוצרו embeddings ל-24 chunks
דוגמה: chunk-0 -> וקטור באורך 768


## שלב 4 – שמירה ב-Pinecone (Vector Database)

**מה זה Pinecone?**
מסד נתונים וקטורי שיודע לחפש במהירות "איזה וקטורים הכי דומים" לשאלה.
ב-RAG, אחרי שיצרנו embeddings לכל chunk – שומרים אותם ב-Pinecone עם metadata (טקסט, עמוד).

**מה קורה כאן?**
1. מתחברים ל-Pinecone עם המפתח מ-`.env`
2. יוצרים index (אם עדיין לא קיים) עם מימד 768 ו-metric `cosine`
3. מעלים (upsert) את כל ה-chunks עם הוקטורים שלהם

In [9]:
from pinecone import Pinecone, ServerlessSpec

if not PINECONE_API_KEY:
    raise ValueError(
        "חסר PINECONE_API_KEY. מלאי אותו בקובץ .env (לא ב-.env.example)."
    )

pc = Pinecone(api_key=PINECONE_API_KEY)

# יצירת index רק אם הוא לא קיים
existing_indexes = {idx.name for idx in pc.list_indexes()}
if INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBED_DIM,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print(f"נוצר index חדש: {INDEX_NAME}")
else:
    print(f"Index '{INDEX_NAME}' כבר קיים – משתמשים בו")

index = pc.Index(INDEX_NAME)
print("מחוברים ל-Pinecone index:", INDEX_NAME)

Index 'rag-apartment-guide' כבר קיים – משתמשים בו
מחוברים ל-Pinecone index: rag-apartment-guide


In [10]:
# הכנת records ל-upsert: (id, vector, metadata)
records = [
    (
        chunk["id"],
        chunk["vector"],
        {
            "text": chunk["text"][:1000],  # Pinecone מגביל metadata – שומרים תחילת הטקסט
            "page": chunk["page"],
        },
    )
    for chunk in chunks
]

# upsert = הוספה/עדכון וקטורים ב-index
index.upsert(vectors=records)

stats = index.describe_index_stats()
print(f"הועלו {len(records)} vectors")
print("סטטיסטיקות index:", stats)

הועלו 24 vectors
סטטיסטיקות index: DescribeIndexStatsResponse(dimension=768, total_vector_count=24, metric='cosine', namespaces=1)


## שלב 5 – Semantic Search (חיפוש סמנטי)

**איך זה עובד?**
1. לוקחים שאלה של המשתמש
2. יוצרים embedding לשאלה עם `RETRIEVAL_QUERY`
3. שולחים את הוקטור ל-Pinecone ומבקשים את 3 התוצאות הקרובות ביותר (`top_k=3`)
4. בודקים אם התוצאות באמת רלוונטיות לשאלה

In [12]:
def semantic_search(question, top_k=3):
    """מחפש את top_k הקטעים הרלוונטיים ביותר לשאלה."""
    query_vector = embed_texts([question], task_type="RETRIEVAL_QUERY")[0]
    results = index.query(
        vector=query_vector,
        top_k=top_k,
        include_metadata=True,
    )
    return results.matches


def print_results(question, matches):
    print("=" * 70)
    print("שאלה:", question)
    print("=" * 70)
    for i, match in enumerate(matches, start=1):
        page = match.metadata.get("page", "?")
        text = match.metadata.get("text", "")
        print(f"\n#{i} | score: {match.score:.4f} | page: {page} | id: {match.id}")
        print(text[:350], "...")

In [13]:
# 5 שאלות לדוגמה על המדריך
questions = [
    "מה צריך לבדוק לגבי היתר בנייה לפני רכישת דירה?",
    "איך מבטיחים את כספי הרוכש לפי חוק המכר?",
    "מה קורה אם המוכר מאחר במסירת הדירה?",
    "מה לבדוק בחוזה המכר לפני חתימה?",
    "מהי תקופת הבדק ומה אחריות המוכר?",
]

for q in questions:
    matches = semantic_search(q, top_k=3)
    print_results(q, matches)
    print()

שאלה: מה צריך לבדוק לגבי היתר בנייה לפני רכישת דירה?

#1 | score: 0.7832 | page: 7 | id: chunk-6
הדירה את הקרקע 
המוכרובתמורה מקבל חלק מהדירות בבניין שיבנה, עליכם לבדוק: האם רשומה הערת אזהרה על שם 
בקשר המוכרקע, ומהם תנאי הﬠִסקה בין קרקע, האם יוכל הקונה לרשום הערת אזהרה על הקרל לבין 
קרקע המקוריים.בעל הקרקע. שימו לב שהדירה העומדת לרכישה לא יוחדה למי מבעלי ה 
היתר בנייה 
שימו לב שהדירה המועמדת לרכישה תיבנה כדין. ודאו כי בידי המוכר היתר בנייה שה ...

#2 | score: 0.7371 | page: 5 | id: chunk-4
יצע דירות קיימות בבניין
נוחות המגורים שלכם. לדוגמה: דירה בקומה ראשונה, קלה לגישה אך היא קרובה יותר לקרקע וחשופה 
בדרך כלל למעבר של דיירים רבים. יש לבדוק כמה קומות יש בבניין ומהי מידת הפרטיות שמספקת כל 
דירה.
6 
 :2פרק
 בודקיםמה לפני הרכישה
מתבצעת על ידי קבלן רשום כחוקהאם הבנייה ?
של המוכר להציג בפניכם רישיון מרשום בפנקס הקבלנים. בקשו הודאו שהדירה נ ...

#3 | score: 0.7350 | page: 3 | id: chunk-1
עש באזור 
o אווירהכיווני בדירה מיקום זריחת השמש ושקיעתהו 
o ביבההס נשקפתה הדירהתוך מ 
• הפרויקט 
o קבלן ה

## בונוס – השוואת `chunk_size` ו-`chunk_overlap`

**למה זה חשוב?**
גודל ה-chunk משפיע ישירות על איכות השליפה:
- **chunks גדולים** – יותר הקשר, אבל פחות ממוקדים; עלולים "לדלל" את הרלוונטיות.
- **chunks קטנים** – ממוקדים יותר, אבל עלולים לפספס הקשר רחב.
- **overlap גבוה** – שומר על רצף בין קטעים, אבל יוצר chunks כפולים.

**מה נשווה?**
| הגדרה | chunk_size | chunk_overlap |
|-------|-----------|---------------|
| A (ברירת מחדל) | 1000 | 150 |
| B (chunks קטנים) | 500 | 80 |

נריץ את **אותה שאלת בדיקה** על שתי ההגדרות ונשווה את 3 התוצאות הראשונות.
(ההשוואה כאן מקומית עם cosine similarity – בלי Pinecone, כדי לא ליצור index שני.)

In [ ]:
import numpy as np

CONFIGS = {
    "A_default": {"chunk_size": 1000, "chunk_overlap": 150},
    "B_small":   {"chunk_size": 500,  "chunk_overlap": 80},
}

TEST_QUESTION = "מה קורה אם המוכר מאחר במסירת הדירה?"


def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def search_local(question, chunk_list, top_k=3):
    """חיפוש מקומי: embed שאלה + chunks, מחזיר top_k לפי cosine similarity."""
    q_vec = embed_texts([question], task_type="RETRIEVAL_QUERY")[0]
    doc_vecs = embed_texts([c["text"] for c in chunk_list], task_type="RETRIEVAL_DOCUMENT")

    scored = []
    for chunk, vec in zip(chunk_list, doc_vecs):
        scored.append((cosine_similarity(q_vec, vec), chunk))

    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_k]

In [ ]:
comparison_results = {}

for name, cfg in CONFIGS.items():
    cfg_chunks = chunk_text(full_text, char_page, **cfg)
    top = search_local(TEST_QUESTION, cfg_chunks, top_k=3)
    comparison_results[name] = top

    print("=" * 70)
    print(f"הגדרה {name}: chunk_size={cfg['chunk_size']}, overlap={cfg['chunk_overlap']}")
    print(f"מספר chunks: {len(cfg_chunks)}")
    print("=" * 70)

    for i, (score, chunk) in enumerate(top, start=1):
        print(f"\n#{i} | score: {score:.4f} | page: {chunk['page']} | id: {chunk['id']}")
        print(chunk["text"][:300], "...")
    print()

### מסקנות (מלאי אחרי הרצה)

**מה לצפות לראות:**
- בהגדרה B (chunks קטנים) – **יותר chunks** נוצרים, וה-top result לרוב **ממוקד יותר** סביב נושא הפיצויים/איחור.
- בהגדרה A (chunks גדולים) – פחות chunks, לפעמים התוצאה כוללת **יותר הקשר** (מידע נוסף שלא קשור ישירות לשאלה).
- overlap גבוה יותר (150 vs 80) – מפחית סיכוי שמשפט ייחתך באמצע, במחיר של chunks מיותרים.

**תיעוד קצר (ערכי אחרי שרצת):**

| | הגדרה A (1000/150) | הגדרה B (500/80) |
|---|---|---|
| מספר chunks | ? | ? |
| score של תוצאה #1 | ? | ? |
| האם התוצאה רלוונטית? | ? | ? |

> **המלצה לפרויקט:** `chunk_size=1000, overlap=150` מתאים למסמך PDF בעברית באורך בינוני.
> אם השליפה "מרחיבה" מדי – נסי להקטין ל-600–800.